In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import yaml
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq

In [ ]:
from src.knn import EnhancedKNNClassifier
from sklearn.decomposition import PCA

In [ ]:
with open("../Lung_Xenium/config_dataset.yaml", "r") as stream:
    config_dataset = yaml.safe_load(stream)
samples = config_dataset["SAMPLE_LQ"]
genes_of_interest = config_dataset["genes_of_interest"]
samples

In [ ]:
ref_lung_atlas = sc.read_h5ad("/cluster/customapps/biomed/grlab/users/knonchev/lung_atlas/b351804c-293e-4aeb-9c4c-043db67f4540.h5ad")
ref_lung_atlas.var.index = ref_lung_atlas.var.feature_name.values
ref_lung_atlas

In [ ]:
for i in ref_lung_atlas.obs.ann_level_2.unique():
    print(i)

In [ ]:
adata_genes = pd.read_csv("../Lung_Xenium/out_benchmark/info_highly_variable_genes.csv").query("isPredicted == True").gene_name.values
shared_genes = np.array(list(set(ref_lung_atlas.var.feature_name.values) & set(adata_genes)))
len(shared_genes)

In [ ]:
ref_lung_atlas = ref_lung_atlas[:, shared_genes].copy()
sc.pp.filter_cells(ref_lung_atlas, min_genes=10)
ref_lung_atlas.X = ref_lung_atlas.X.toarray()

In [ ]:
sc.pp.pca(ref_lung_atlas)
sc.pp.neighbors(ref_lung_atlas)
sc.tl.umap(ref_lung_atlas)

In [ ]:
model = "DeepCell"
adatas_predicted = []
adatas_gt = []
for sample in tqdm(samples):

    adata_path_gt = f"../Lung_Xenium/data/h5ad/{sample}.h5ad"
    adata_gt = sc.read_h5ad(adata_path_gt)
    adata_gt = adata_gt[:, shared_genes].copy()
    sc.pp.normalize_total(adata_gt, target_sum=10000)
    sc.pp.log1p(adata_gt)
    sc.tl.ingest(adata_gt, ref_lung_atlas, embedding_method=('umap', 'pca'))
    adatas_gt.append(adata_gt)

    adata_path_pred = f"../Lung_Xenium/out_benchmark/prediction/{model}/data/h5ad/{sample}.h5ad"
    adata_pred = sc.read_h5ad(adata_path_pred)
    adata_pred = adata_pred[:, shared_genes].copy()
    sc.tl.ingest(adata_pred, ref_lung_atlas, embedding_method=('umap', 'pca'))
    adatas_predicted.append(adata_pred)

In [ ]:
adata_concat = ad.concat([ref_lung_atlas, *adatas_predicted], label="Modality", 
                         keys=["Single-cell", *[f"H&E {s}" for s in samples]])
adata_concat

In [ ]:
clf = KNeighborsClassifier(n_jobs=-1, n_neighbors=15)
clf.fit(ref_lung_atlas.obsm["X_pca"], ref_lung_atlas.obs.ann_level_2)

In [ ]:
adata_concat.obs["label"] = clf.predict(adata_concat.obsm["X_pca"])

In [ ]:
with plt.rc_context({
    "figure.figsize": (10, 8),
    "figure.dpi": 300,
#    "font.size": 17,  # Increase font size
    "axes.titlesize": 17,
    "axes.labelsize": 17,
#    "legend.fontsize": 17,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17
}):

    fig = sc.pl.umap(
        adata_concat, 
        color=["Modality", "label"],
        hspace=7,
        size=3,
        show=False,
        frameon=False,
        return_fig=True
    )

    # Custom titles for each subplot
    custom_titles = ["Integrated single-cell atlas and H&E slides", "Single-cell annotations"]
    for ax, title in zip(fig.axes, custom_titles):
        ax.set_title(title, fontsize=18)
        ax.set_xlabel("")  # Hide x-axis label
        ax.set_ylabel("")  # Hide y-axis label

    #fig.savefig("figures/Lung_Xenium_ref_atlas_umap.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
samples

In [ ]:
adata = adatas[1]
adata.obs["label"] = clf.predict(adata.obsm["X_pca"])

In [ ]:
plt.rcParams.update({'font.size': 14})


bounds = (adata.obsm["spatial"][:, 0].min(),
              adata.obsm["spatial"][:, 1].min()+50,
              adata.obsm["spatial"][:, 0].max()-50,
              adata.obsm["spatial"][:, 1].max()-350)
bounds

In [ ]:
sq.pl.spatial_scatter(adata, 
                      color="label", 
                      title="Transferred single-cell annotations",
                      #img_alpha=0.5,
                      img=False,
                      crop_coord=bounds, 
                      wspace=0.1, 
                      hspace=0.1,
                      size=3,      
                      ncols=1, 
                      cmap="viridis",
                      #title=title, 
                      save=f"figures/Figure7B_lung_xenium_NCBI867_labels.png", 
                      dpi=150,
                      frameon=False, 
                      colorbar=False, 
                      #legend_loc="lower left",
                      legend_fontsize=15,
                      figsize=(7, 7))
plt.show()

In [ ]:
sq.pl.spatial_scatter(adata, 
                      color="label", 
                      title="H&E image",
                      #img_alpha=0.5,
                      img=True,
                      crop_coord=bounds, 
                      wspace=0.1, 
                      hspace=0.1,
                      size=0,      
                      ncols=1, 
                      cmap="viridis",
                      #title=title, 
                      save=f"figures/Figure7B_lung_xenium_NCBI867_image.png", 
                      dpi=150,
                      frameon=False, 
                      colorbar=False, 
                      #legend_loc="lower left",
                      legend_fontsize=15,
                      figsize=(7, 7))
plt.show()

In [ ]:
adata_concat_pred = ad.concat(adatas_predicted, label="Modality", 
                         keys=[f"H&E {s}" for s in samples])
adata_concat_pred

In [ ]:
adata_concat_gt = ad.concat(adatas_gt, label="Modality", 
                         keys=[f"H&E {s}" for s in samples])
adata_concat_gt

In [ ]:
clf = RandomForestClassifier(n_jobs=-1, max_depth=25)#KNeighborsClassifier(n_jobs=-1, n_neighbors=50)
clf.fit(ref_lung_atlas.X, ref_lung_atlas.obs.ann_level_2)
clf.score(ref_lung_atlas.X, ref_lung_atlas.obs.ann_level_2)

In [ ]:
adata_concat_pred.obs["label"] = clf.predict(adata_concat_pred.X)
adata_concat_gt.obs["label"] = clf.predict(adata_concat_gt.X)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Actual and predicted labels
y_true = adata_concat_pred.obs["label"]
y_pred = adata_concat_gt.obs["label"]

# Get unique class labels
labels = np.unique(y_true)

# Compute normalized confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=labels, normalize='true')
cm = np.round(cm, 2)

# Compute absolute counts of each true label
class_counts = np.array([(y_true == label).sum() for label in labels])

# Append the counts column
cm_with_counts = np.hstack([cm, class_counts.reshape(-1, 1)])

# Define tick labels
xticklabels = list(labels) + ['Total']
yticklabels = list(labels)

# Create the mask for the last column
mask = np.zeros_like(cm_with_counts, dtype=bool)
mask[:, -1] = True  # Mask last column for coloring

# Plot the heatmap (with masked coloring for last column)
plt.figure(figsize=(8, 6))
ax = sns.heatmap(cm_with_counts, annot=False, fmt='g', cmap='Blues', cbar=False,
                 xticklabels=xticklabels, yticklabels=yticklabels, mask=mask)

# Add all annotations manually, including last column
for i in range(cm_with_counts.shape[0]):
    for j in range(cm_with_counts.shape[1]):
        val = cm_with_counts[i, j]
        ax.text(j + 0.5, i + 0.5, f'{val:.0f}' if j == cm_with_counts.shape[1]-1 else f'{val:.2f}',
                ha='center', va='center', color='black', fontsize=10)

plt.title('Confusion Matrix (Normalized) with Class Counts')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import balanced_accuracy_score
balanced_accuracy_score(y_true, y_pred)